# Task 1

In [19]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

X = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])

y_and = np.array([0, 0, 0, 1])
y_xor = np.array([0, 1, 1, 0])


def predict(X, w, b):
    z = X @ w + b
    return (z > 0).astype(int)


def train_perceptron(X, y, eta=0.5, max_epochs=20):
    w = np.zeros(2)
    b = 0.0
    mistakes_per_epoch = []

    for epoch in range(max_epochs):
        mistakes = 0
        for i in range(len(X)):
            y_hat = 1 if (np.dot(w, X[i]) + b) > 0 else 0 
            err = y[i] - y_hat
            if err != 0:
                mistakes += 1
            w = w + eta * err * X[i]
            b = b + eta * err
        mistakes_per_epoch.append(mistakes)
        if mistakes == 0:
            break

    return w, b, mistakes_per_epoch


w_and, b_and, hist_and = train_perceptron(X, y_and)
w_xor, b_xor, hist_xor = train_perceptron(X, y_xor)

print("AND: epochs =", len(hist_and), "w =", w_and, "b =", b_and)
print("AND: mistakes per epoch =", hist_and)
print("AND: final predictions  =", predict(X, w_and, b_and))
print()
print("XOR: epochs =", len(hist_xor), "w =", w_xor, "b =", b_xor)
print("XOR: mistakes per epoch =", hist_xor)
print("XOR: final predictions  =", predict(X, w_xor, b_xor))
print()

wrong_and = int(np.sum(predict(X, w_and, b_and) != y_and))
wrong_xor = int(np.sum(predict(X, w_xor, b_xor) != y_xor))
print("FINAL model errors (fixed parameters, 4 inputs):")
print("  AND: %d / 4 wrong" % wrong_and)
print("  XOR: %d / 4 wrong" % wrong_xor)
print("  last-epoch online mistakes: AND =", hist_and[-1], " XOR =", hist_xor[-1])

plt.figure(figsize=(7, 4.5))
plt.plot(range(1, len(hist_and) + 1), hist_and, "o-", label="AND")
plt.plot(range(1, len(hist_xor) + 1), hist_xor, "s-", label="XOR")
plt.xlabel("Epoch")
plt.ylabel("Online mistakes during the epoch")
plt.title("Online mistakes per epoch (counted while w, b keep changing)\nNOT an evaluation of the final model")
plt.xticks(range(1, max(len(hist_and), len(hist_xor)) + 1))
plt.yticks(range(0, 5))
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("mistakes_per_epoch.png", dpi=150)

assert np.all(predict(X, w_and, b_and) == y_and)
print("\nTask 1 OK")

AND: epochs = 6 w = [1.  0.5] b = -1.0
AND: mistakes per epoch = [1, 3, 3, 2, 1, 0]
AND: final predictions  = [0 0 0 1]

XOR: epochs = 20 w = [-0.5  0. ] b = 0.5
XOR: mistakes per epoch = [2, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
XOR: final predictions  = [1 1 0 0]

FINAL model errors (fixed parameters, 4 inputs):
  AND: 0 / 4 wrong
  XOR: 2 / 4 wrong
  last-epoch online mistakes: AND = 0  XOR = 4

Task 1 OK


1.	Why must the parameters be reset before the XOR run?
2.	Why does success on AND not imply success on XOR?
3.	Why can more epochs not overcome a single perceptron’s linear-boundary limitation on XOR?
4.	Are online mistakes during an epoch always equal to the final model’s errors? Explain.


1) If the state is transferred between runs, the result begins to depend on which notebook cells you launched first. If you relaunched in a different order, you'll get different numbers, and it's unclear why.
2) The perceptron convergence theorem guarantees a finite number of updates only for linearly separable data. XOR is not covered by the theorem's conditions, so it makes no promises about it.
3) Question 2 showed that the perceptron can't solve XOR. Searching any longer is pointless: you'll just end up traversing more incorrect lines.
4) Online error is the error the model made before updating the weights in this example.
After training is complete, we fix the final (w,b) values ​​and run them through the entire dataset again. So there are not equal

# Task 2

### Part A

In [20]:
import numpy as np

x = np.array([0.5, 1.0, 1.5, 2.0])
y = np.array([2.0, 3.0, 4.0, 5.0])

params0 = np.array([0.5, 0.5, 1.0, 0.0])


In [21]:
def forward(params, x):
    w1, b1, w2, b2 = params

    z1 = w1 * x + b1
    h = np.maximum(0.0, z1)
    yhat = w2 * h + b2

    return z1, h, yhat


def batch_loss(params, x, y):
    z1, h, yhat = forward(params, x)
    assert yhat.shape == y.shape
    return np.mean((yhat - y) ** 2)

z1, h, yhat = forward(params0, x)
se = (yhat - y) ** 2

print(f"{'x':>5} {'y':>5} {'z1':>7} {'h':>7} {'yhat':>7} {'yhat-y':>8} {'sq.err':>9}")
for i in range(len(x)):
    print(f"{x[i]:5.2f} {y[i]:5.2f} {z1[i]:7.2f} {h[i]:7.2f} "
          f"{yhat[i]:7.2f} {yhat[i]-y[i]:8.2f} {se[i]:9.4f}")

print("\nsum of squared errors =", se.sum())
print("mean loss L           =", batch_loss(params0, x, y))

assert np.allclose(yhat, [0.75, 1.0, 1.25, 1.5])

    x     y      z1       h    yhat   yhat-y    sq.err
 0.50  2.00    0.75    0.75    0.75    -1.25    1.5625
 1.00  3.00    1.00    1.00    1.00    -2.00    4.0000
 1.50  4.00    1.25    1.25    1.25    -2.75    7.5625
 2.00  5.00    1.50    1.50    1.50    -3.50   12.2500

sum of squared errors = 25.375
mean loss L           = 6.34375


### Part B

$$\frac{\partial L}{\partial \hat{y}^{(i)}}
= \frac{\partial}{\partial \hat{y}^{(i)}} \left[ \frac{1}{N} \sum_{j=1}^{N} (\hat{y}^{(j)} - y^{(j)})^2 \right]
= \frac{2}{N} \left( \hat{y}^{(i)} - y^{(i)} \right).$$


$$\frac{\partial L}{\partial w_2} = \sum_{i=1}^{N} \frac{\partial L}{\partial \hat{y}^{(i)}} \cdot h^{(i)},
\qquad
\frac{\partial L}{\partial b_2} = \sum_{i=1}^{N} \frac{\partial L}{\partial \hat{y}^{(i)}} \cdot 1 .$$

$$\frac{\partial L}{\partial h^{(i)}} = \frac{\partial L}{\partial \hat{y}^{(i)}} \cdot w_2 .$$


$$\mathrm{ReLU}'(z) = \begin{cases} 1, & z > 0 \\ 0, & z \le 0 \end{cases}
\qquad\Longrightarrow\qquad
\frac{\partial L}{\partial z_1^{(i)}} = \frac{\partial L}{\partial h^{(i)}} \cdot \mathbf{1}\!\left[z_1^{(i)} > 0\right].$$

$$\frac{\partial L}{\partial w_1} = \sum_{i=1}^{N} \frac{\partial L}{\partial z_1^{(i)}} \cdot x^{(i)},
\qquad
\frac{\partial L}{\partial b_1} = \sum_{i=1}^{N} \frac{\partial L}{\partial z_1^{(i)}} \cdot 1 .$$


In [22]:
def batch_gradients(params, x, y):
    w1, b1, w2, b2 = params

    z1, h, yhat = forward(params, x)
    N = len(x)

    g_yhat = 2.0 * (yhat - y) / N
    g_w2 = np.sum(g_yhat * h)
    g_b2 = np.sum(g_yhat)
    g_h = g_yhat * w2
    g_z1 = g_h * (z1 > 0).astype(float)
    g_w1 = np.sum(g_z1 * x)
    g_b1 = np.sum(g_z1)

    return np.array([g_w1, g_b1, g_w2, g_b2])

In [23]:
g = batch_gradients(params0, x, y)
print("analytic  =", g)

eps = 1e-6
num = np.zeros(4)
for k in range(4):
    pp = params0.copy(); pp[k] += eps
    pm = params0.copy(); pm[k] -= eps
    num[k] = (batch_loss(pp, x, y) - batch_loss(pm, x, y)) / (2 * eps)

print("numerical =", num)
print("max diff  =", np.max(np.abs(g - num)))

analytic  = [-6.875  -4.75   -5.8125 -4.75  ]
numerical = [-6.875  -4.75   -5.8125 -4.75  ]
max diff  = 9.083862551051425e-10


1. Where does the factor 1/N enter the gradients, and why?
2. Why do the shared-parameter gradients require a sum over the examples?
3. What is the difference between the per-example vectors of shape (4,) and the returned parameter-gradient vector of shape (4,)?
4. Why is the output of this model not a probability?

1) Two is the derivative of the square, 1/N is the averaging in the loss. Then this multiplier is simply carried along the chain unchanged, and does not appear anywhere again.
2) Because the parameters are the same for all four examples. If I change w1 slightly, all four predictions will change at once, meaning the loss will change due to all four at once. According to the chain rule, contributions from different paths are added together..
3) It just happened that there are 4 examples and 4 parameters as well. If there were 10 examples, the first ones would be (10,), and the final one would still be (4,).
4) Because the output is simply ŷ, there's no sigmoid at all. This could yield either -5 or 100, but the probability must lie between 0 and 1.


# Task 3

### Part A

In [24]:
def numerical_gradient(fn, params, eps=1e-5):

    gradient = np.zeros_like(params, dtype=float)

    for j in range(len(params)):

        plus = params.copy()
        minus = params.copy()

        plus[j] += eps     
        minus[j] -= eps

        gradient[j] = (fn(plus) - fn(minus)) / (2 * eps)

    return gradient


analytic = batch_gradients(params0, x, y)

numeric = numerical_gradient(
    lambda p: batch_loss(p, x, y), params0
)

print("analytic:", analytic)
print("numeric :", numeric)
print("max absolute difference:", np.max(np.abs(analytic - numeric)))

assert np.max(np.abs(analytic - numeric)) < 1e-6

analytic: [-6.875  -4.75   -5.8125 -4.75  ]
numeric : [-6.875  -4.75   -5.8125 -4.75  ]
max absolute difference: 8.662937034387141e-11


### Part B

In [25]:
params = params0.copy()
eta = 0.01

old_params = params.copy()
old_loss = batch_loss(params, x, y)

gradients = batch_gradients(params, x, y) 
params = params - eta * gradients

new_loss = batch_loss(params, x, y)

names = ["w1", "b1", "w2", "b2"]
print(f"{'param':>6} | {'old':>10} | {'gradient':>10} | {'-eta*grad':>10} | {'new':>10}")
print("-" * 58)
for n, o, g, p in zip(names, old_params, gradients, params):
    print(f"{n:>6} | {o:>10.6f} | {g:>10.6f} | {-eta*g:>10.6f} | {p:>10.6f}")

print()
print(f"old loss : {old_loss:.6f}")
print(f"new loss : {new_loss:.6f}")
print(f"change   : {new_loss - old_loss:+.6f}")

 param |        old |   gradient |  -eta*grad |        new
----------------------------------------------------------
    w1 |   0.500000 |  -6.875000 |   0.068750 |   0.568750
    b1 |   0.500000 |  -4.750000 |   0.047500 |   0.547500
    w2 |   1.000000 |  -5.812500 |   0.058125 |   1.058125
    b2 |   0.000000 |  -4.750000 |   0.047500 |   0.047500

old loss : 6.343750
new loss : 5.109202
change   : -1.234548


In [26]:
params = params0.copy()
eta = 0.01

loss_history = [batch_loss(params, x, y)]

for step in range(200):

    gradients = batch_gradients(params, x, y)
    params = params - eta * gradients

    loss_history.append(batch_loss(params, x, y))

print("final params:", params)
print("final loss  :", loss_history[-1])

assert len(loss_history) == 201
assert loss_history[-1] < loss_history[0]

final params: [1.21710839 0.65019193 1.54577988 0.1613669 ]
final loss  : 0.004726306469450738


In [27]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(range(len(loss_history)), loss_history, color="tab:blue")
axes[0].set_xlabel("update number")
axes[0].set_ylabel("batch loss $L$")
axes[0].set_title("Loss vs update (linear scale)")
axes[0].grid(alpha=0.3)

axes[1].semilogy(range(len(loss_history)), loss_history, color="tab:red")
axes[1].set_xlabel("update number")
axes[1].set_ylabel("batch loss $L$ (log scale)")
axes[1].set_title("Same curve, log scale")
axes[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print(f"loss at update   0 : {loss_history[0]:.6f}")
print(f"loss at update   1 : {loss_history[1]:.6f}")
print(f"loss at update  50 : {loss_history[50]:.6f}")
print(f"loss at update 100 : {loss_history[100]:.6f}")
print(f"loss at update 200 : {loss_history[200]:.6f}")
print(f"number of stored values: {len(loss_history)}")

loss at update   0 : 6.343750
loss at update   1 : 5.109202
loss at update  50 : 0.036618
loss at update 100 : 0.018719
loss at update 200 : 0.004726
number of stored values: 201


C:\Users\Admin\AppData\Local\Temp\ipykernel_37476\2780222149.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


1.	Which function performs backpropagation? Which statement updates the parameters?
2.	Why is updating a weight before finishing the backward pass incorrect?
3.	Does the reduced loss on these four training examples demonstrate performance on unseen data? Explain.

1) Backprop creates batch_gradients. It first calls forward to get z1, h, and ŷ, and then goes backwards: first it calculates g_yhat, then g_h, then g_z1, and along the way it collects gradients for all four parameters.
2) Because all gradients must be calculated at the same point, with the same parameter values. Therefore, the order is: first, fully forward, then fully backward, and only then change all four parameters at once.
3) No. We calculated the loss using the same four data points we trained on. This only shows that the model has learned/adapted to them, not that it can handle anything new. To test the model on new data, we need separate examples that weren't used in training at all.